# 6. PAR Model with Normalized Covariation (NCV)


## Load the derived residual series

This notebook uses the residual series created by `02_decomposition.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

residual_series = pd.read_csv(
    "data/residual_series.csv",
    parse_dates=["date"],
    index_col="date"
)["residual"].dropna()

print(f"Loaded {len(residual_series)} residual observations.")


## Concepts

The NCV approach uses **normalized covariation** in PAR estimation.

### Analogy
When two people have very different activity levels, normalizing their relationship can make the comparison less dependent on raw scale. This notebook applies the NCV-based PAR procedure to the same residual series.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === INPUT: residual series ===
y = residual_series.values
index = residual_series.index

# === Robust MAD function ===
def mad(x):
    return np.median(np.abs(x - np.median(x))) + 1e-8  # add epsilon to avoid zero-div

# === Normalized Covariation function (robust) ===
def ncv_autocorr(series, lag):
    if lag == 0:
        return 1.0
    x = series[:-lag]
    y = series[lag:]
    return np.median(x * y) / (mad(x) * mad(y))

# === Yule-Walker style equation using NCV ===
def ncv_yule_walker(series, L):
    r = [ncv_autocorr(series, h) for h in range(L + 1)]  # r[0], r[1], ..., r[L]
    R = np.array([[r[abs(i - j)] for j in range(L)] for i in range(L)])
    r_vec = np.array(r[1:L+1])
    try:
        phi = np.linalg.solve(R, r_vec)
    except np.linalg.LinAlgError:
        phi = np.zeros(L)
    return phi

# === Fit PAR(p, L) using NCV method ===
def fit_par_ncv(y, p, L):
    n = len(y)
    phi = {}
    fitted = np.zeros_like(y)
    
    for s in range(p):
        # Phase series: {y_t | t mod p = s}
        y_s = np.array([y[t] for t in range(p * L, n) if t % p == s])
        if len(y_s) <= L + 1:
            phi[s] = np.zeros(L)
            continue
        phi[s] = ncv_yule_walker(y_s, L)

    for t in range(p * L, n):
        s = t % p
        if s in phi:
            lag_vec = y[t - L:t][::-1]
            fitted[t] = np.dot(phi[s], lag_vec)

    return phi, fitted

# === AIC Computation ===
def compute_par_ncv_aic(y, p, L):
    n = len(y)
    phi, fitted = fit_par_ncv(y, p, L)
    residuals = y[p * L:] - fitted[p * L:]
    rss = np.sum(residuals ** 2)
    T = len(residuals)
    if T == 0 or rss == 0:
        return np.inf
    aic = T * np.log(rss / T) + 2 * (p * L)
    return aic

# === Search for best (p, L) ===
best_aic = np.inf
best_p = None
best_L = None

for p in range(2, 31, 2):  # Try periods: 2, 4, ..., 30
    for L in range(1, 4):  # Try AR orders: 1, 2, 3
        aic = compute_par_ncv_aic(y, p, L)
        if aic < best_aic:
            best_aic = aic
            best_p = p
            best_L = L

# === Final Model Fit with Best (p, L) ===
phi, fitted = fit_par_ncv(y, best_p, best_L)

# === Plot Actual vs Predicted ===
plt.figure(figsize=(12, 4))
plt.plot(index, y, label='Actual Residuals', alpha=0.6)
plt.plot(index, fitted, label=f'PAR({best_L}) Fit with NCV (p={best_p})', color='purple')
plt.title('PAR Model Fit using Normalized Covariation (NCV)')
plt.xlabel('Time')
plt.ylabel('Residual Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# === Report Best Parameters ===
print(f"✅ Best Period (p): {best_p}")
print(f"✅ Best Order (L): {best_L}")
print(f"✅ Minimum AIC (NCV): {best_aic:.2f}")

